# W07 — Content Action Playbook: Decline-Risk Triage

This notebook turns the held-out Week-5 predictions into a human-reviewed investigation queue. It is decision support, not an automated publishing or refresh system. Run Weeks 5 and 6 first. The target is a proxy: a ≥20% decrease in clicks per measured day in the later March half. It does not prove why a decline happened or that an edit will reverse it.

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'work').exists(): ROOT = Path('/content/FlyRank_Internship')
OUT = ROOT / 'work' / 'outputs'
FIG = ROOT / 'work' / 'figures'; FIG.mkdir(parents=True, exist_ok=True)
pred_path, metrics_path = OUT/'w05_test_predictions.csv', OUT/'w05_metrics.json'
assert pred_path.exists() and metrics_path.exists(), 'Run Week 5 first.'
pred = pd.read_csv(pred_path); metrics = json.loads(metrics_path.read_text())
assert {'client_hash_id','content_hash_id','decline_label','logistic_score','forest_score','baseline_score'}.issubset(pred.columns)
print('Held-out pages:', len(pred))

## 1. Ranked actions and reason codes

Use Logistic Regression only if it is at least as good as the fixed baseline at Precision@20. Otherwise retain the rule. Reason codes explain review priority; they are not guarantees.

In [ ]:
comparison = pd.DataFrame(metrics['comparison']).set_index('method')
model_p20 = float(comparison.loc['Logistic Regression','precision_at_20'])
baseline_p20 = float(comparison.loc['Week-4 rule baseline','precision_at_20'])
use_model = model_p20 >= baseline_p20
pred['selected_score'] = pred['logistic_score'] if use_model else pred['baseline_score']
pred['selected_method'] = 'Logistic Regression' if use_model else 'Week-4 rule baseline'
pct = pred['selected_score'].rank(pct=True)
pred['risk_tier'] = np.select([pct>=.8,pct>=.5],['priority_review','review'],default='monitor')
pred['reason_code'] = pred['risk_tier'].map({'priority_review':'high_relative_decline_priority','review':'moderate_relative_decline_priority','monitor':'monitor_or_low_relative_priority'})
pred['recommended_action'] = pred['risk_tier'].map({'priority_review':'check coverage, query mix, seasonality and SERP changes','review':'review GSC trend and content context','monitor':'recheck next reporting window'})
queue = pred.sort_values('selected_score',ascending=False).copy(); queue.insert(0,'review_rank',np.arange(1,len(queue)+1))
display(queue[['review_rank','risk_tier','reason_code','recommended_action','selected_method','selected_score','decline_label']].head(20))
print('Selected:', 'model' if use_model else 'baseline', '| Precision@20:', model_p20, baseline_p20)

## 2. Operational archetype → action mapping

These are operational triage groups, not semantic content clusters.

| Archetype | Evidence pattern | Suggested human action |
|---|---|---|
| Priority decline review | Top 20% of selected ranking | Check measurement, query mix, seasonality, SERP changes, then inspect content |
| Model/rule disagreement | Rankings differ materially | Escalate for manual review; inspect which early signals explain the difference |
| Monitor | Lower-ranked eligible pages | Recheck next reporting period |

No archetype establishes that content age caused a decline.

In [ ]:
pred['baseline_rank'] = pred['baseline_score'].rank(method='first',ascending=False)
pred['model_rank'] = pred['logistic_score'].rank(method='first',ascending=False)
pred['rank_gap'] = (pred['baseline_rank']-pred['model_rank']).abs()
pred['archetype'] = np.select([pred['risk_tier'].eq('priority_review'),pred['rank_gap']>=max(10,int(.1*len(pred)))],['priority_decline_review','model_rule_disagreement'],default='monitor')
summary = pred.groupby('archetype').agg(pages=('content_hash_id','size'),observed_decline_proxy_rate=('decline_label','mean'),median_rank_gap=('rank_gap','median')).reset_index()
display(summary)

## 3. Intended use, limits, human review and no-go list

**Intended use:** help an SEO/content reviewer decide which eligible items to investigate first.

**Human review:** verify GSC coverage; inspect query-level mix privately; check seasonality, SERP layout, position and tracking changes; then review page intent and content before proposing an edit.

**Do not automate:** title/meta edits, deletion, canonical changes, redirects, client communications or budget decisions. Never publish pseudonymous IDs, private queries, raw URLs or client details.

**Limits:** one March evaluation window; proxy label; client holdout but no multi-month temporal test; observational data; no causal evidence for refresh impact.

## 4. Monitoring and value

Track Precision@10/20 and base rate on a later untouched month, GSC coverage, score/rank stability, reviewer overrides and later outcomes where an approved feedback process exists. Re-evaluate when schema/label definitions change, coverage shifts, ranking falls below baseline, or a new untouched month becomes available. False positives consume reviewer time; false negatives delay investigation. No monetary ROI is claimed because costs and intervention outcomes are not measured.

## 5. Exports for the paper

The pseudonymous operational queue stays local and must not be committed or published. Only aggregate public-safe figures are suitable for the paper.

In [ ]:
import matplotlib.pyplot as plt
private_cols = ['review_rank','client_hash_id','content_hash_id','archetype','risk_tier','reason_code','recommended_action','selected_method','selected_score','logistic_score','forest_score','baseline_score','decline_label']
private_path = OUT/'w07_private_action_queue.csv'; queue[private_cols].to_csv(private_path,index=False)
public = queue.groupby('risk_tier').agg(page_count=('content_hash_id','size'),observed_decline_proxy_rate=('decline_label','mean')).reset_index()
display(public)
ax = public.plot(x='risk_tier',y='page_count',kind='bar',legend=False,figsize=(7,4),title='Held-out review queue by triage tier')
ax.set_xlabel('Triage tier'); ax.set_ylabel('Eligible pages')
plt.tight_layout(); plt.savefig(FIG/'w07_queue_tiers.png',dpi=160); plt.show()
receipt = {'selected_method':'Logistic Regression' if use_model else 'Week-4 rule baseline','selection_metric':'Precision@20','model_precision_at_20':model_p20,'baseline_precision_at_20':baseline_p20,'heldout_pages':int(len(queue)),'archetypes':summary.to_dict(orient='records'),'public_summary':public.to_dict(orient='records'),'public_figure':'work/figures/w07_queue_tiers.png','limitations':['single March evaluation','proxy outcome','no causal intervention evidence','no multi-month temporal validation']}
(OUT/'w07_playbook_metrics.json').write_text(json.dumps(receipt,indent=2))
print('Private queue:',private_path,'(do not commit)')
print('Public-safe figure:',FIG/'w07_queue_tiers.png')

## 6. Self-check

- [ ] Selected method is justified by held-out Precision@20.
- [ ] Actions are human investigation recommendations, not automated edits.
- [ ] Archetypes are labeled operational, not semantic clusters.
- [ ] Limits and no-go cases are explicit.
- [ ] Private queue is not committed or published.
- [ ] Only aggregate public-safe figures are committed.